# 12 - DCA vs ML Technical Recovery Comparison

Notebook 10 created DCA technical modeled recoverable oil estimates.

Notebook 11 created recursive ML technical recovery scenarios.

This notebook merges those outputs into one 30-well comparison table.

Important framing: these are technical modeled recoverable oil estimates through the stated forecast horizon. They are not SPE PRMS reserves estimates.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"
ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

DCA_30_WELL_FILE = DCA_OUTPUT_DIR / "dca_30_well_technical_recoverable_oil_table.csv"
ML_30_WELL_FILE = ML_OUTPUT_DIR / "ml_30_well_technical_recoverable_oil_table.csv"

DCA_30_WELL_FILE.exists(), ML_30_WELL_FILE.exists()

## Load the 30-well tables

The two input files already contain one row per well.

DCA and ML use the same historical cumulative oil through month 33.

In [ ]:
dca_table = pd.read_csv(DCA_30_WELL_FILE)
ml_table = pd.read_csv(ML_30_WELL_FILE)

dca_table.shape, ml_table.shape

In [ ]:
dca_table.head()

In [ ]:
ml_table.head()

## Merge DCA and ML scenarios

We keep a single target label column named `reservoir_or_conduit_target`.

In this dataset, that target is `SPRABERRY (TREND AREA)` for all wells.

In [ ]:
ml_table = ml_table.rename(columns={"field_name": "reservoir_or_conduit_target"})

merge_keys = [
    "api8",
    "well_name",
    "lease_name",
    "well_no",
    "reservoir_or_conduit_target",
    "historical_cumulative_oil_bbl",
]

technical_recovery_comparison = dca_table.merge(
    ml_table,
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

technical_recovery_comparison.shape

In [ ]:
technical_recovery_comparison.head()

## Add simple comparison columns

For a readable first-pass comparison, we use hyperbolic DCA as the main DCA reference and linear regression as the main ML reference.

This does not mean either is the final preferred method. It simply makes one side-by-side delta easy to inspect.

In [ ]:
technical_recovery_comparison["ml_minus_hyperbolic_dca_bbl"] = (
    technical_recovery_comparison["linear_regression_ml_technical_recoverable_oil_bbl"]
    - technical_recovery_comparison["hyperbolic_technical_recoverable_oil_bbl"]
)

technical_recovery_comparison["ml_to_hyperbolic_dca_ratio"] = (
    technical_recovery_comparison["linear_regression_ml_technical_recoverable_oil_bbl"]
    / technical_recovery_comparison["hyperbolic_technical_recoverable_oil_bbl"]
)

technical_recovery_comparison.head()

In [ ]:
display_columns = [
    "api8",
    "well_name",
    "reservoir_or_conduit_target",
    "historical_cumulative_oil_bbl",
    "exponential_technical_recoverable_oil_bbl",
    "hyperbolic_technical_recoverable_oil_bbl",
    "harmonic_technical_recoverable_oil_bbl",
    "linear_regression_ml_technical_recoverable_oil_bbl",
    "naive_last_rate_technical_recoverable_oil_bbl",
    "trailing_3mo_technical_recoverable_oil_bbl",
    "trailing_6mo_technical_recoverable_oil_bbl",
    "ml_minus_hyperbolic_dca_bbl",
    "ml_to_hyperbolic_dca_ratio",
]

technical_recovery_comparison[display_columns].round(2).head(10)

## Cohort-level scenario totals

In [ ]:
scenario_columns = [
    "exponential_technical_recoverable_oil_bbl",
    "hyperbolic_technical_recoverable_oil_bbl",
    "harmonic_technical_recoverable_oil_bbl",
    "linear_regression_ml_technical_recoverable_oil_bbl",
    "naive_last_rate_technical_recoverable_oil_bbl",
    "trailing_3mo_technical_recoverable_oil_bbl",
    "trailing_6mo_technical_recoverable_oil_bbl",
]

cohort_scenario_totals = (
    technical_recovery_comparison[scenario_columns]
    .sum()
    .rename("technical_recoverable_oil_bbl")
    .reset_index()
    .rename(columns={"index": "scenario"})
)

cohort_scenario_totals["technical_recoverable_oil_mmbbl"] = (
    cohort_scenario_totals["technical_recoverable_oil_bbl"] / 1_000_000
)

cohort_scenario_totals.round(2)

## First-pass interpretation

The ML scenarios cluster near the lower end of the DCA range.

In this run, the recursive ML and simple trailing-rate scenarios are close to the exponential DCA total and below the hyperbolic and harmonic DCA totals.

That pattern is reasonable for a first long-horizon ML extension. The ML scenarios are trained on short early-life data and recursively feed predictions back into the feature history, which tends to dampen the long tail.

DCA remains the more natural framework for duration-style recovery forecasting because the decline models explicitly encode a long-tail production shape. The ML scenarios are useful as a sensitivity check, not as a replacement for DCA.

In [ ]:
cohort_scenario_totals.sort_values(
    "technical_recoverable_oil_bbl",
    ascending=False,
).round(2)

## Export comparison outputs

In [ ]:
COMPARISON_OUTPUT_FILE = ML_OUTPUT_DIR / "dca_ml_30_well_technical_recovery_comparison.csv"
COMPARISON_TOTALS_FILE = ML_OUTPUT_DIR / "dca_ml_technical_recovery_scenario_totals.csv"

COMPARISON_OUTPUT_FILE, COMPARISON_TOTALS_FILE

In [ ]:
technical_recovery_comparison[display_columns].to_csv(COMPARISON_OUTPUT_FILE, index=False)
cohort_scenario_totals.to_csv(COMPARISON_TOTALS_FILE, index=False)

In [ ]:
COMPARISON_OUTPUT_FILE.exists(), COMPARISON_TOTALS_FILE.exists()

## Notebook 12 summary

This notebook creates one comparison table for the 30 wells.

The table places DCA and ML technical recoverable oil scenarios side by side.

These values are not reserves estimates. No SPE PRMS classification, economic limit, or abandonment cutoff has been applied.